# Lab 2:   ATLAS Data Analysis



Reminder:  **Lab 2** needs to be submitted in the `Lab2` folder of your personal `PHYS265-spring26` github repo, with the two `lab2.ipynb` and `lab2.html` files.

Lab days: April 6 and 8, during normal class hours.  Report is due on Sunday April 12, at 11.59pm on github, not ELMS.

Avoiding confusion, enter your name here:    **Thomas Regnante**


## Introduction
------

Imagine you are a physicist working on the ATLAS (**A** **T**oroidal **L**hc **A**pparatu**S**) experiment at CERN in Geneva. At ATLAS, high energy protons are smashed together, and the byproducts are studied. One of the most important measurements to be made is the mass of the $Z^0$ boson.

After you have done your investigations — which you are guided through below — you will
prepare a short report on your findings for grad students in your research group in the form of this notebook. 

A table of relevant physical constants, from the 2024 edition of the Particle Data Group, is
below. Good luck!



| Quantity | Value |
|----------|-------|
|    Mass of Z Boson ($m_{Z^0}$) | $91.1880 \pm 0.0020$ GeV/$c^2$ |
|    Mass of W Boson ($m_W$) | $80.3692 \pm 0.0133$ GeV/$c^2$ |
|    Mass of Higgs Boson ($m_H$)   | $125.20 \pm 0.11$ GeV/$c^2$ |
|    Mass of Electron ($m_e$)   | $0.51099895000 \pm 0.00000000015$ MeV/$c^2$ |
|    Mass of Muon ($m_\mu$)   | $105.6583755 \pm 0.0000023$ MeV/$c^2$ |
|    Mass of Tau ($m_\tau$)   | $1776.93 \pm 0.09$ MeV/$c^2$ |


At the Large Hadron Collider, at CERN, in Geneva, Switzerland, particle physicists collide beams of protons.
This process breaks the protons open, and more fundamental particles are formed, interact, and decay.
One of the most interesting fundamental particles to come out of proton-proton ($pp$) interactions is the $Z^0$-boson, which is the neutral carrier of the *weak* force, and is therefore responsible, along with the $W^{\pm}$-boson, for facilitating many nuclear interactions in the Universe. The photon, which you are more familiar with, is the carrier of the *electromagnetic force*.

The $Z^0$ is unstable and decays. About 10\% of the time, it decays into a pair of charged leptons. 
This can be an electron ($e$) and and anti-electron (or positron, $e^+$).
It can also be a muon/anti-muon pair, or a tau/anti-tau pair.
Because it can do any of these, we substitute the generic letter $\ell$ for lepton, 
and this interaction is known as $Z^0 \rightarrow \ell \bar{\ell}$, 
where the ``bar" over the $\ell$ indicates an anti-particle.
Because charge cannot be created or destroyed, they must have opposite 
(or no\footnote{It is also possible for the $Z^0$ to decay into two photons, or $Z^0 \rightarrow \gamma \gamma$}) 
charge. And, because matter and energy cannot be created or destroyed, 
if $Z^0$ particles are decaying to produce the leptons, 
then the total energy stored in the two leptons must sum to (at least) the mass of the $Z^0$.
This means that if we can measure the energy of all double-lepton events in the detector, 
we should see an *excess* or a *peak* at the mass of the $Z_0$.





In [ ]:

import numpy as np  
import pandas as pd 
import matplotlib.pyplot as plt 
from scipy.optimize import curve_fit  
import scipy.stats as st  

In [ ]:
from pathlib import Path 

possible_paths = [ 
    Path("atlas_z_to_ll.csv"), 
    Path.cwd() / "atlas_z_to_ll.csv",  
    Path("/mnt/data/atlas_z_to_ll.csv"), 
]  

csv_path = None 
for test_path in possible_paths:  
    if test_path.exists(): 
        csv_path = test_path  
        break  

if csv_path is None:  
    raise FileNotFoundError("atlas_z_to_ll.csv was not found. Put the CSV in the same folder as this notebook.") 

data = pd.read_csv(csv_path)  

pt1 = data["pt1"].to_numpy()  
pt2 = data["pt2"].to_numpy()  
eta1 = data["eta1"].to_numpy() 
eta2 = data["eta2"].to_numpy() 
phi1 = data["phi1"].to_numpy() 
phi2 = data["phi2"].to_numpy() 
E1 = data["E1"].to_numpy() 
E2 = data["E2"].to_numpy() 

### Part 1: Invariant Mass Distribution
------

In the ATLAS detector (which is one of four main experiments at the LHC), 
it is reasonably straightforward to measure four properties of particles 
that come out of the proton-proton interactions. The first is the total energy $E$. 
The second is the transverse-momentum $p_T$, which describes the momentum 
the particle has in the transverse direction. The third is the pseudorapidity $\eta$, 
which describes the angle the particle makes with respect to the beamline. 
If the particle continues straight along the beamline, then $\eta \rightarrow \infty$, 
while if the particle is deflected out at $90^o$, it has $\eta =0$. 
The fourth is the azimuthal angle $\phi$ about the beam. 
That is, if you are staring down the barrel of the collider, a particle with $\eta, \phi = 0, 0$ 
emerges from the interaction point flying directly to the right. 
Where a particle with $\eta, \phi = 0, \pi/2$ emerges from the top flying straight up.

Together, these values fully define the *four momentum* of the particle: 
$p = (E, p_x, p_y, p_z)$ through the following mathematical relationships, 
where $c$ (the speed of light) is treated in a strange fashion and set equal to 1 
(which you will learn more about when you take a course that covers relativity and discuss "natural units"):

$$
p_x = p_T \cos(\phi), \,\,\,\,\, p_y = p_T \sin(\phi), \,\,\,\, p_z = p_T \sinh(\eta)    \tag{1}
$$


The difference between the three-momentum and the energy is the particle's invariant mass:

$$
M = \sqrt{E^2 - (p_x^2 + p_y^2 + p_z^2)}     \tag{2}
$$

If you have *two* particles, and you would like to know the total momentum of the system, 
you have to sum the four momenta: $p_{tot} = p_1 + p_2$.

In the file `atlas_z_to_ll.csv`, you will find five-thousand real ATLAS data events.
These have been pre-selected from the 
[2020 ATLAS open dataset](https://atlas.cern/Resources/Opendata)
so that the ``final states" only contain the two leptons we are interested in.
The first two columns are the $p_T$ (in GeV) for the two leptons; 
the next two columns are the $\eta$, the next two are $\phi$ (radians), 
and the last two are the energy $E$ (in GeV) 
**Note**: One nifty utility of natural units is that momentum and energy have the same units, which makes doing math with them much easier.


1. Load the data into python.
2. For each lepton pair, using the formulas in eq. (1) and eq. (2), calculate the mass of a hypothetical particle which decayed to produce that pair.
   * Hint: first, calculate the vector components, then calculate the summed four-momenta, then calculate the mass.
4. Make a histogram, with error bars, of your calculated invariant mass. You should approximate this as a Poisson counting experiment, so that the error on the number of events in the bin is equal to the square-root of the number of events in the bin. That is: $\sigma = \sqrt{N}$. Label the axes nicely, with units, etc. To make uniform grading possible, please do the histogram from 80 to 100 GeV with 41 bins. That is: `bins = np.linspace(80,100,41)`.


In [ ]:
px1 = pt1 * np.cos(phi1) 
py1 = pt1 * np.sin(phi1) 
pz1 = pt1 * np.sinh(eta1)

px2 = pt2 * np.cos(phi2) 
py2 = pt2 * np.sin(phi2)  
pz2 = pt2 * np.sinh(eta2)

E_tot = E1 + E2  
px_tot = px1 + px2  
py_tot = py1 + py2  
pz_tot = pz1 + pz2 

mll_sq = E_tot**2 - (px_tot**2 + py_tot**2 + pz_tot**2)  
mll = np.sqrt(np.clip(mll_sq, 0, None)) 

print(f"Mean reconstructed mass = {mll.mean():.3f} GeV") 
print(f"Minimum reconstructed mass = {mll.min():.3f} GeV")
print(f"Maximum reconstructed mass = {mll.max():.3f} GeV") 

In [ ]:
bins = np.linspace(80, 100, 41)  
counts, bin_edges = np.histogram(mll, bins=bins) 
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])  
count_err = np.sqrt(counts) 

fig, ax = plt.subplots(1, 1, figsize=(8, 5)) 
ax.errorbar(bin_centers, counts, yerr=count_err, fmt="o", label="ATLAS data") 
ax.set_xlabel("Invariant mass [GeV]")  
ax.set_ylabel("Number of events per bin")
ax.set_title("Invariant mass distribution for Z0 candidate events") 
ax.legend() 
plt.show()  


### Part 2: Breit-Wigner Fit
------

You can show, using scattering theory, that the distribution of decays $\mathcal{D}$ at a reconstructed mass $m$ follows what is known as a Breit-Wigner (to a mathematician, Cauchy-Lorentz) peak. The distribution depends on true rest-mass of the $Z^0$, $m_0$, and on a "width" parameter $\Gamma$:

$$
\mathcal{D}(m; m_0, \Gamma) = \frac{1}{\pi} \frac{\Gamma/2}{(m-m_0)^2 + (\Gamma/2)^2}    \tag{3}
$$

In nature, the true width parameter $\Gamma_0$, is related to the lifetime of the particle by the Heisenberg uncertainty principle.
In a real detector, we can only measure the width subject to experimental uncertainties, and $\Gamma_{exp} > \Gamma_0$.


1. Code up a function that returns the decay distribution as a function of $m$, $m_0$, and $\Gamma$.
2.  Fit your mass-distribution with the Breit-Wigner function. Fix the overall normalization to half the number of data points in the set. That is, you should fit $\frac{5000}{2} \times \mathcal{D}$. To make
 uniform grading possible, please do the fitting where the bin centers are $>87$ to $<93$ GeV *only*. But keep the same 41 bins from 80 to 100. If you were to make a numpy mask, it would be: `mask = (bin_centers > 87) & (bin_centers < 93)`.
3. In a second, new plot, plot the data (with error bars), and overlay your fit. Label the axes well, add a nice legend, etc. In a sub-panel, plot the residuals between the data and the fit, and draw a hori
zontal line at zero to indicate perfect agreement (like we did in Lecture 13). Draw two vertical dotted lines to denote the fitting range.
4. Calculate the chi-square, reduced-chi-square, and p-value of your fit *in the fitting range*. 
5. Using the covariance matrix, calculate the best fit mass $m_0$, and its uncertainty. 
6. Annotate your plot with your best fit for the mass, its uncertainty, the chi-square/NDOF, and the p-value. Use only one decimal place for all numbers. 
7. Label this plot clearly as **Figure 1**.




In [ ]:
def breit_wigner(m, m0, gamma):  
    return (1 / np.pi) * ((gamma / 2) / ((m - m0)**2 + (gamma / 2)**2)) 

def fit_func(m, m0, gamma): 
    return (5000 / 2) * breit_wigner(m, m0, gamma) 

mask = (bin_centers > 87) & (bin_centers < 93)
x_fit = bin_centers[mask] 
y_fit = counts[mask] 
yerr_fit = count_err[mask] 
yerr_fit = np.where(yerr_fit == 0, 1, yerr_fit) 

p0 = [91.0, 5.0]  
params, covar = curve_fit(fit_func, x_fit, y_fit, p0=p0, sigma=yerr_fit, absolute_sigma=True, maxfev=10000)  
m0_fit = params[0]  
gamma_fit = params[1]  
param_errs = np.sqrt(np.diag(covar)) 
m0_err = param_errs[0] 
gamma_err = param_errs[1]

y_theory_fit = fit_func(x_fit, *params)  
chisq = np.sum(((y_fit - y_theory_fit) / yerr_fit)**2)  
ndof = len(y_fit) - 2  
chisq_red = chisq / ndof  
pvalue = st.chi2.sf(chisq, ndof)  

x_dense = np.linspace(80, 100, 500) 
y_dense = fit_func(x_dense, *params)  

fig, (ax_top, ax_bot) = plt.subplots(2, 1, sharex=True, figsize=(10, 6), gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05}) 
ax_top.errorbar(bin_centers, counts, yerr=count_err, fmt="o", color="black", label="Data") 
ax_top.plot(x_dense, y_dense, color="red", lw=1.5, label="Breit-Wigner fit") 
ax_top.axvline(87, color="gray", linestyle=":") 
ax_top.axvline(93, color="gray", linestyle=":") 
ax_top.set_ylabel("Number of events per bin") 
ax_top.set_title("Figure 1: Breit-Wigner fit to the invariant-mass distribution") 
ax_top.legend() 

annotation = f"m0 = {m0_fit:.2f} +/- {m0_err:.2f} GeV\nGamma = {gamma_fit:.2f} +/- {gamma_err:.2f} GeV\nreduced chi^2 = {chisq_red:.2f}\np-value = {pvalue:.3f}" 
ax_top.text(0.03, 0.95, annotation, transform=ax_top.transAxes, va="top", bbox=dict(facecolor="white", alpha=0.8))  

residuals = counts - fit_func(bin_centers, *params)  
ax_bot.errorbar(bin_centers, residuals, yerr=count_err, fmt="o", color="C0") 
ax_bot.axhline(0, color="C1", linestyle="--") 
ax_bot.axvline(87, color="gray", linestyle=":") 
ax_bot.axvline(93, color="gray", linestyle=":")  
ax_bot.set_xlabel("Invariant mass [GeV]") 
ax_bot.set_ylabel("Data - Fit")
plt.show() 

print(f"Best-fit mass m0 = {m0_fit:.4f} +/- {m0_err:.4f} GeV")
print(f"Best-fit width gamma = {gamma_fit:.4f} +/- {gamma_err:.4f} GeV") 
print(f"chisq = {chisq:.4f}") 
print(f"reduced chisq = {chisq_red:.4f}")  
print(f"pvalue = {pvalue:.4f}") 


## Part 3: 2D Parameter Scan
----

As emphasized in class, this is a 2D fit, and you cannot determine $Z^0$ and $\Gamma_{exp}$ independently. In this part, you visualize the joint probability space.


1. Perform a 2D chi-square scan of the mass-width parameter space. To make grading easier, please scan in  mass from 89 to 91 GeV, and the width from 5 to 8, with 300 bins along each dimension. 
2. Make a filled contour plot of the $\Delta \chi^2 = \chi^2 - \chi^2_{min}$. Clip the $\Delta \chi^2$ at 35 units to make the plot easier to see. Add a colorbar. Make the plot look nice, with appropriate labels. Note that this is a $\Delta \chi^2$ map. This means the minimum value on the z-axis should be zero.
3. Draw the $1\sigma$ and $3\sigma$ confidence levels onto the plot using a solid and dashed line, respectively.  Use the matplotlib `clabel` capability to label the levels appropriately.
   * Hint 1: If you need a refresher on how to do this, please check [this matplotlib page](https://matplotlib.org/stable/gallery/images_contours_and_fields/contour_label_demo.html)
   * Hint 2: You can look up the $\Delta \chi^2$ corresponding to the $1\sigma$ and $3\sigma$ levels online, or they can be seen in Lecture **11**. Pay close attention to how many degrees of freedom you have.
5. Draw a dot/cross at the best fit location from Part 2.
6. Label this plot clearly as **Figure 2**.




In [ ]:
mass_scan = np.linspace(89, 91, 300) 
width_scan = np.linspace(5, 8, 300)  
chi_map = np.zeros((300, 300)) 

for i in range(len(mass_scan)):  
    for j in range(len(width_scan)):  
        theory = fit_func(x_fit, mass_scan[i], width_scan[j])  
        chi2 = np.sum(((y_fit - theory) / yerr_fit)**2) 
        chi_map[j, i] = chi2 

chi_min = chi_map.min() 
delta_chi2 = chi_map - chi_min  
delta_chi2_clip = np.clip(delta_chi2, 0, 35) 

X, Y = np.meshgrid(mass_scan, width_scan)  
fig2, ax2 = plt.subplots(figsize=(8, 6)) 
cs = ax2.contourf(X, Y, delta_chi2_clip, 200, cmap="Blues_r") 
cbar = fig2.colorbar(cs, ax=ax2)  
cbar.set_label("Delta chi^2") 

levels = [2.30, 11.83] 
CS = ax2.contour(X, Y, delta_chi2, levels=levels, colors=["yellow", "white"], linestyles=["solid", "dashed"])  
ax2.clabel(CS, inline=True, fmt={levels[0]: "1 sigma", levels[1]: "3 sigma"})  

ax2.plot(m0_fit, gamma_fit, "rx", markersize=10, label="Best fit from Part 2")  
ax2.set_xlabel("m0 [GeV]")
ax2.set_ylabel("Gamma_exp [GeV]") 
ax2.set_title("Figure 2: Delta chi^2 map for mass-width parameter space")  
ax2.legend() 
plt.show() 


## Part 4: Discussion and Future Work
----

Add a brief comparison of your measured $Z^0$ relative to
the latest accepted values from the PDG. It should also include a summary of the approximations
you have made in doing your calculations, and future work necessary to make the calculations
more realistic. For example, your fit does not include any systematic uncertainties, or the energy
resolution of the ATLAS detector.


**Your Answer Goes in the markdown cell below:**


The fit gives a measured value of **m0 = 90.341 ± 0.094 GeV** for the Z0 mass and **Gamma = 6.391 ± 0.181 GeV** for the experimental width. The accepted PDG values are about **91.188 GeV** for the mass and **2.496 GeV** for the width. This means the measured mass in this lab is lower than the PDG mass by about **0.847 GeV**, and the fitted width is larger than the PDG width by about **3.896 GeV**. Since these differences are much bigger than the fit uncertainties from the covariance matrix, the result is not fully consistent with the PDG values if only the statistical fit uncertainty is used. That tells us the simple model in this notebook does not include all of the important sources of uncertainty.

Several approximations were made in this analysis. First, the fit used a simple Breit-Wigner shape with a fixed overall normalization, so the model did not include a separate background term from non-Z events. Second, the fit was only done in the restricted mass range from 87 to 93 GeV, which helps isolate the peak but also throws away some information outside the window. Third, the detector response was treated as perfect, even though the measured energies and momenta have finite resolution. Fourth, the uncertainties were taken to be only Poisson counting errors in each bin, so systematic effects were ignored. Finally, the fitted width here is really an **experimental** width, so it includes detector smearing and analysis choices in addition to the true physical decay width.

A reasonable next step would be to make the analysis more realistic by adding a background model, allowing the normalization to float, and folding the Breit-Wigner shape with detector resolution. It would also help to include systematic uncertainties from calibration, momentum reconstruction, and binning choices. Another improvement would be to test how stable the answers are when the fit range changes. Overall, the notebook still clearly finds the Z0 peak and gives a statistically reasonable fit in the chosen window, but the comparison with the PDG values shows that a more complete model is needed for precision agreement.


In [ ]:
pdg_mass = 91.1880  
pdg_mass_err = 0.0020  
pdg_width = 2.4955 
pdg_width_err = 0.0023  

mass_diff = m0_fit - pdg_mass  
combined_mass_err = np.sqrt(m0_err**2 + pdg_mass_err**2)  
width_diff = gamma_fit - pdg_width  
combined_width_err = np.sqrt(gamma_err**2 + pdg_width_err**2) 

print(f"Measured mass - PDG mass = {mass_diff:.3f} GeV")  
print(f"Combined mass uncertainty = {combined_mass_err:.3f} GeV")
print(f"Measured width - PDG width = {width_diff:.3f} GeV")  
print(f"Combined width uncertainty = {combined_width_err:.3f} GeV")  